In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, VBox, interactive_output
from IPython.display import display


In [2]:
def plot_space_vector_diagram(ax, angle_rad, modulation_index):
    """
    Draws a space vector diagram illustrating V_ref as the sum of the 3 phase vectors.
    - Uses alpha/beta axes.
    - Shows Va, Vb, Vc as vectors.
    - Shows V_ref as the vector sum of Va, Vb, Vc.
    """
    # Define the six base vectors of the SVM hexagon
    v_hexagon = [
        np.array([1, 0]),
        np.array([np.cos(np.pi/3), np.sin(np.pi/3)]),
        np.array([np.cos(2*np.pi/3), np.sin(2*np.pi/3)]),
        np.array([-1, 0]),
        np.array([np.cos(4*np.pi/3), np.sin(4*np.pi/3)]),
        np.array([np.cos(5*np.pi/3), np.sin(5*np.pi/3)])
    ]
    
    # Draw the SVM hexagon frame
    for i in range(6):
        # Draw the outer edges of the hexagon
        ax.plot([v_hexagon[i][0], v_hexagon[(i+1)%6][0]], 
                [v_hexagon[i][1], v_hexagon[(i+1)%6][1]], 
                'k-', alpha=0.3) # Solid black line, slightly transparent
    
    # --- Setup and Base Vectors ---
    # Define the directions of the phase vectors
    phase_a_vec_dir = np.array([1, 0])
    phase_b_vec_dir = np.array([np.cos(2*np.pi/3), np.sin(2*np.pi/3)]) # 120 degrees
    phase_c_vec_dir = np.array([np.cos(4*np.pi/3), np.sin(4*np.pi/3)]) # 240 degrees

    # Draw the axes for the phases
    ax.plot([0, phase_a_vec_dir[0]], [0, phase_a_vec_dir[1]], 'r--', alpha=0.5)
    ax.text(1.1, 0, 'Va', color='r', fontsize=12, ha='center')
    ax.plot([0, phase_b_vec_dir[0]], [0, phase_b_vec_dir[1]], 'g--', alpha=0.5)
    ax.text(phase_b_vec_dir[0]*1.1, phase_b_vec_dir[1]*1.1, 'Vb', color='g', fontsize=12, ha='center')
    ax.plot([0, phase_c_vec_dir[0]], [0, phase_c_vec_dir[1]], 'b--', alpha=0.5)
    ax.text(phase_c_vec_dir[0]*1.1, phase_c_vec_dir[1]*1.1, 'Vc', color='b', fontsize=12, ha='center')

    # --- Clarke Transform Visualization ---
    # Calculate the instantaneous amplitude of each phase voltage
    # Note: We use cosine here to align with the standard d-q frame where d is on the horizontal axis
    va_inst = modulation_index * np.cos(angle_rad)
    vb_inst = modulation_index * np.cos(angle_rad - 2*np.pi/3)
    vc_inst = modulation_index * np.cos(angle_rad + 2*np.pi/3)

    # Create the three phase vectors by scaling the direction vectors by the instantaneous amplitudes
    va_vec = va_inst * phase_a_vec_dir
    vb_vec = vb_inst * phase_b_vec_dir
    vc_vec = vc_inst * phase_c_vec_dir

    # The reference vector is the sum of these three vectors (this is the Clarke Transform)
    # The 2/3 scaling factor is part of the amplitude-invariant Clarke Transform
    v_ref_vec = (2/3) * (va_vec + vb_vec + vc_vec)

    # --- Draw the Vectors ---
    # Draw the individual phase vectors
    ax.arrow(0, 0, va_vec[0], va_vec[1], head_width=0.05, head_length=0.08, fc='r', ec='r', alpha=0.7, label=f'Va ({va_inst:.2f})')
    ax.arrow(0, 0, vb_vec[0], vb_vec[1], head_width=0.05, head_length=0.08, fc='g', ec='g', alpha=0.7, label=f'Vb ({vb_inst:.2f})')
    ax.arrow(0, 0, vc_vec[0], vc_vec[1], head_width=0.05, head_length=0.08, fc='b', ec='b', alpha=0.7, label=f'Vc ({vc_inst:.2f})')
    
    # Draw the final reference vector
    ax.arrow(0, 0, v_ref_vec[0], v_ref_vec[1], head_width=0.06, head_length=0.1, fc='k', ec='k', length_includes_head=True, label='V_ref (Sum)')

    # --- Configure the plot ---
    limit = 1.1
    ax.set_xlim([-limit, limit]); ax.set_ylim([-limit, limit])
    ax.set_aspect('equal', adjustable='box')
    ax.set_title('1. V_ref as Sum of Phase Vectors (Clarke Transform)')
    ax.set_xlabel('α-axis'); ax.set_ylabel('β-axis')
    ax.grid(True)
    ax.legend(fontsize='small')

In [3]:
# --- 2. Main plotting function ---
# --- Main plotting function ---
def plot_svm(angle_deg, modulation_index):
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    fig.tight_layout(pad=5.0)
    
    ax1 = axs[0, 0]; ax4 = axs[0, 1]; ax2 = axs[1, 0]; ax3 = axs[1, 1]
    angle_rad = np.deg2rad(angle_deg)

    # --- Call Space Vector Diagram function ---
    plot_space_vector_diagram(ax1, angle_rad, modulation_index)

    # --- Instantaneous 3-Phase Sine Waves Plot (ax4) ---
    # (No changes here)
    time_domain = np.linspace(0, 360, 500); rad_domain = np.deg2rad(time_domain)
    va = modulation_index * np.cos(rad_domain); vb = modulation_index * np.cos(rad_domain - 2*np.pi/3); vc = modulation_index * np.cos(rad_domain + 2*np.pi/3)
    ax4.plot(time_domain, va, 'r', label='Va'); ax4.plot(time_domain, vb, 'g', label='Vb'); ax4.plot(time_domain, vc, 'b', label='Vc')
    ax4.axvline(x=angle_deg, color='k', linestyle='--', label=f'Angle: {angle_deg}°')
    va_inst = modulation_index * np.cos(angle_rad); vb_inst = modulation_index * np.cos(angle_rad - 2*np.pi/3); vc_inst = modulation_index * np.cos(angle_rad + 2*np.pi/3)
    ax4.plot(angle_deg, va_inst, 'ro', markersize=8); ax4.plot(angle_deg, vb_inst, 'go', markersize=8); ax4.plot(angle_deg, vc_inst, 'bo', markersize=8)
    ax4.set_title('2. Instantaneous 3-Phase Values'); ax4.set_xlabel('Electrical Angle (°)')
    ax4.set_ylabel('Amplitude'); ax4.set_xlim([0, 360]); ax4.set_ylim([-1.1, 1.1]); ax4.grid(True); ax4.legend(fontsize='small')

    # --- MODIFIED: PWM Duty Cycle Waveform Plot (ax2) ---
    # We will now calculate the duty cycles for a full 360-degree sweep
    T_pwm = 1.0; sqrt3 = np.sqrt(3)
    angles_sweep = np.linspace(0, 2*np.pi, 360)
    duty_a, duty_b, duty_c = [], [], []

   # --- SVM Duty Cycle Calculation (for both plots) ---
    T_pwm = 1.0; sqrt3 = np.sqrt(3)
    
    # --- For the waveform plot (ax2) ---
    angles_sweep = np.linspace(0, 2*np.pi, 360)
    duty_a_wave, duty_b_wave, duty_c_wave = [], [], []
    for ang in angles_sweep:
        sector = int(np.floor(np.rad2deg(ang)) / 60) % 6 + 1
        gamma = ang - (sector - 1) * np.pi / 3
        T1 = sqrt3 * T_pwm * modulation_index/2 * np.sin(np.pi/3 - gamma)
        T2 = sqrt3 * T_pwm * modulation_index/2 * np.sin(gamma)
        T0 = T_pwm - T1 - T2
        if T0 < 0: T1, T2, T0 = T1+T0/2, T2+T0/2, 0
        
        sector_times = {1: (T1+T2+T0/2, T2+T0/2, T0/2), 
                        2: (T1+T0/2, T1+T2+T0/2, T0/2), 
                        3: (T0/2, T1+T2+T0/2, T2+T0/2), 
                        4: (T0/2, T1+T0/2, T1+T2+T0/2), 
                        5: (T2+T0/2, T0/2, T1+T2+T0/2), 
                        6: (T1+T2+T0/2, T0/2, T1+T0/2)}
        
        Ta_on, Tb_on, Tc_on = sector_times[sector]
        duty_a_wave.append(Ta_on / T_pwm); 
        duty_b_wave.append(Tb_on / T_pwm); 
        duty_c_wave.append(Tc_on / T_pwm)

    ax2.plot(np.rad2deg(angles_sweep), duty_a_wave, 'r', label='Duty A'); 
    ax2.plot(np.rad2deg(angles_sweep), duty_b_wave, 'g', label='Duty B'); 
    ax2.plot(np.rad2deg(angles_sweep), duty_c_wave, 'b', label='Duty C')
    
    ax2.axvline(x=angle_deg, color='k', linestyle='--'); ax2.set_title('3. SVM Duty Cycles vs. Angle')
    ax2.set_xlabel('Electrical Angle (°)'); ax2.set_ylabel('Duty Cycle (0 to 1)'); ax2.grid(True); ax2.legend(fontsize='small'); ax2.set_xlim([0, 360]); ax2.set_ylim([0, 1])


    # --- NEW: Detailed Single PWM Period Plot (ax3) ---
    # Get the duty cycles for the *current* angle
    duty_a_now = duty_a_wave[int(angle_deg) % 360]
    duty_b_now = duty_b_wave[int(angle_deg) % 360]
    duty_c_now = duty_c_wave[int(angle_deg) % 360]

    # Function to create a centered pulse waveform
    def create_centered_pulse(duty_cycle, y_offset):
        on_time = duty_cycle * T_pwm
        off_time_each_side = (T_pwm - on_time) / 2
        start_time = off_time_each_side
        end_time = start_time + on_time
        # [time_points], [y_values]
        return ([0, start_time, start_time, end_time, end_time, T_pwm], 
                [y_offset, y_offset, y_offset + 1, y_offset + 1, y_offset, y_offset])

    # Create pulses for each phase, stacked vertically
    time_a, wave_a = create_centered_pulse(duty_a_now, 4)
    time_b, wave_b = create_centered_pulse(duty_b_now, 2)
    time_c, wave_c = create_centered_pulse(duty_c_now, 0)

    ax3.plot(time_a, wave_a, 'r', label=f'Phase A Gate (Duty: {duty_a_now:.2f})')
    ax3.plot(time_b, wave_b, 'g', label=f'Phase B Gate (Duty: {duty_b_now:.2f})')
    ax3.plot(time_c, wave_c, 'b', label=f'Phase C Gate (Duty: {duty_c_now:.2f})')
    
    ax3.set_title('4. Inverter Gate Signals (One PWM Period)')
    ax3.set_xlabel('Time (normalized to T_pwm)')
    ax3.set_yticks([0, 0.5, 1, 2, 2.5, 3, 4, 4.5, 5], [0, 'Phase C',1,0, 'Phase B',1,0, 'Phase A',1])
    ax3.set_xlim([0, T_pwm]); ax3.set_ylim([-0.5, 5.5])
    ax3.grid(True); ax3.legend(fontsize='small')
    
    plt.show()

In [4]:

mod_index_max = 2 / np.sqrt(3);

# --- Create the Interactive Sliders ---
angle_slider = FloatSlider(min=0, max=360, step=1, value=45, description='Vector Angle (°):', continuous_update=True, layout={'width': '600px'})
mod_index_slider = FloatSlider(min=0, max=mod_index_max, step=0.05, value=0.8, description='Mod. Index:', continuous_update=True, layout={'width': '600px'})

# --- Link Sliders to the Plotting Function using interactive_output ---
controls = VBox([angle_slider, mod_index_slider])
out = interactive_output(plot_svm, {'angle_deg': angle_slider, 'modulation_index': mod_index_slider})

# --- Display the layout ---
display(VBox([controls, out]))

In [5]:
# --- CORRECTED Core plotting and calculation function ---

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
from ipywidgets import interactive_output, FloatSlider, VBox
from IPython.display import display
def plot_dead_zones(adc_settling_time_us, pwm_frequency_khz):
    """
    This function visualizes the current measurement dead zones in the α-β plane.
    The width of the dead zones is determined by the ADC settling time and PWM frequency.
    """
    # --- 1. Setup the Figure ---
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # --- 2. Calculate Dead Zone Width ---
    T_pwm_us = 1000 / pwm_frequency_khz
    T0_min_us = 2 * adc_settling_time_us
    min_duty_cycle = T0_min_us / T_pwm_us
    dead_zone_angle_deg = (min_duty_cycle / 0.5) * 30
    if dead_zone_angle_deg > 30: dead_zone_angle_deg = 30

    # --- 3. Draw the SVM Hexagon (the green area) ---
    hexagon_vertices = []
    for i in range(7):
        angle = np.deg2rad(60 * i)
        hexagon_vertices.append([np.cos(angle), np.sin(angle)])
    
    # Create the hexagon patch and add it to the plot
    hexagon_patch = patches.Polygon(hexagon_vertices, closed=True, facecolor='mediumseagreen', edgecolor='k', linewidth=2, label='Valid Measurement Region')
    ax.add_patch(hexagon_patch)

    # --- 4. Draw the Dead Zones (the white bands) ---
    for i in range(6):
        center_angle = 60 * i
        start_angle = center_angle - dead_zone_angle_deg
        end_angle = center_angle + dead_zone_angle_deg
        
        dead_zone_wedge = patches.Wedge(
            center=(0, 0), r=1.5, theta1=start_angle, theta2=end_angle, 
            facecolor='white', edgecolor='gray', linestyle='--'
        )
        ax.add_patch(dead_zone_wedge)

    # --- 5. Add Labels and Final Touches ---
    vector_labels = ['V₁₀₀', 'V₁₁₀', 'V₀₁₀', 'V₀₁₁', 'V₀₀₁', 'V₁₀₁']
    for i in range(6):
        angle = np.deg2rad(60 * i)
        x, y = np.cos(angle), np.sin(angle)
        ax.text(x * 1.1, y * 1.1, vector_labels[i], fontsize=14, ha='center', va='center')
        ax.plot([0, x*0.95], [0, y*0.95], 'k--', alpha=0.5)

    # --- FIX IS HERE ---
    # 1. Create a patch for the legend, but DO NOT add it to the axes.
    dead_zone_legend_handle = patches.Patch(color='white', ec='gray', linestyle='--', label=f'Dead Zone ({dead_zone_angle_deg*2:.1f}° wide)')
    
    # 2. Build the list of handles for the legend and call ax.legend().
    legend_handles = [hexagon_patch, dead_zone_legend_handle]
    ax.legend(handles=legend_handles)
    # --- END OF FIX ---
    
    ax.set_title(f'Current Measurement Dead Zones\n(T_pwm = {T_pwm_us:.1f} µs, Min Time = {adc_settling_time_us:.1f} µs)')
    ax.set_xlabel('α-axis'); ax.set_ylabel('β-axis')
    ax.set_xlim([-1.3, 1.3]); ax.set_ylim([-1.3, 1.3])
    ax.set_aspect('equal', adjustable='box'); ax.grid(True)
    plt.show()

# --- Create the Interactive Sliders ---
adc_slider = FloatSlider(
    min=0.5, max=5.0, step=0.1, value=1.5, 
    description='ADC Settle Time (µs):', 
    continuous_update=True, layout={'width': '500px'}
)
pwm_freq_slider = FloatSlider(
    min=10, max=50, step=2, value=20, 
    description='PWM Frequency (kHz):', 
    continuous_update=True, layout={'width': '500px'}
)

# --- Link Sliders to the Plotting Function ---
controls = VBox([adc_slider, pwm_freq_slider])
out = interactive_output(plot_dead_zones, {'adc_settling_time_us': adc_slider, 'pwm_frequency_khz': pwm_freq_slider})

# --- Display the layout ---
display(VBox([controls, out]))

In [6]:
# Import necessary libraries
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

# --- 1. The Core Calculation and Display Function ---
def calculate_limits(t_adc_ns, t_opamp_ns, t_margin_ns, f_sw_khz):
    """
    Calculates and displays the modulation and duty cycle limits
    based on timing and frequency parameters.
    """
    # --- Convert inputs to microseconds ---
    t_adc = t_adc_ns / 1000
    t_opamp = t_opamp_ns / 1000
    t_margin = t_margin_ns / 1000
    
    # --- Perform Calculations from the Text ---
    # Total acquisition time
    t_acq = t_adc + t_opamp
    # Total required time for measurement
    t_req = t_acq + t_margin
    # PWM switching period
    T_sw = 1000 / f_sw_khz # Result is in microseconds
    
    # Maximum modulation index
    # Formula: m <= 1 - (2 * t_req / T_sw)
    m_max = 1 - (2 * t_req / T_sw)
    
    # Clamp m_max to be non-negative for clarity
    if m_max < 0:
        m_max = 0
        
    # Minimum and Maximum Duty Cycles
    # Formula: d_min = t_req / T_sw
    # Formula: d_max = 1 - (t_req / T_sw)
    d_min = t_req / T_sw
    d_max = 1 - d_min
    
    # --- 2. Generate and Display the Output using HTML for rich formatting ---
    
    # Interpretation Text
    if m_max >= 0.97:
        interp_text = "<b>Excellent:</b> You can run essentially up to the linear SVM limit. Only the top ~3% or less of modulation is excluded."
        interp_color = "green"
    elif m_max >= 0.95:
        interp_text = "<b>Good:</b> With modest margins, operation is fully compatible without special PWM shaping."
        interp_color = "blue"
    elif m_max >= 0.90:
        interp_text = "<b>Fair:</b> Operation is possible, but the upper range of modulation is significantly limited."
        interp_color = "orange"
    else:
        interp_text = "<b>Poor:</b> The required settling time is too long for this switching frequency, severely limiting performance."
        interp_color = "red"

    # Build the HTML output string
    output_html = f"""
    <div style="border: 1px solid #ccc; padding: 15px; border-radius: 5px; font-family: sans-serif;">
        <h3 style="margin-top:0;">Calculation Results</h3>
        <p>
            <b>Total Required Time (t<sub>req</sub>):</b>
            {t_acq:.2f} µs (acq) + {t_margin:.2f} µs (margin) = <b>{t_req:.2f} µs</b>
        </p>
        <p>
            <b>PWM Period (T<sub>sw</sub>):</b>
            1000 / {f_sw_khz} kHz = <b>{T_sw:.2f} µs</b>
        </p>
        <hr>
        <div style="background-color: #f0f8ff; padding: 10px; border-radius: 5px;">
            <h4>Maximum Modulation Index (m<sub>max</sub>)</h4>
            <p style="font-size: 24px; font-weight: bold; margin: 5px 0 10px 0;">
                m<sub>max</sub> = 1 - (2 * {t_req:.2f} / {T_sw:.2f}) = <span style="color: #0056b3;">{m_max:.3f}</span>
            </p>
        </div>
          

        <div style="display: flex; justify-content: space-between;">
            <div style="flex-basis: 48%;">
                <h4>Maximum Duty Cycle (d<sub>max</sub>)</h4>
                <p style="font-size: 18px;">1 - ({t_req:.2f} / {T_sw:.2f}) = <b>{d_max:.3f} ({d_max*100:.1f}%)</b></p>
            </div>
            <div style="flex-basis: 48%;">
                <h4>Minimum Duty Cycle (d<sub>min</sub>)</h4>
                <p style="font-size: 18px;">{t_req:.2f} / {T_sw:.2f} = <b>{d_min:.3f} ({d_min*100:.1f}%)</b></p>
            </div>
        </div>
        <hr>
        <h4>Interpretation</h4>
        <p style="color: {interp_color};">{interp_text}</p>
    </div>
    """
    
    # Display the HTML
    display(HTML(output_html))


# --- 2. Create the Interactive Sliders ---
style = {'description_width': 'initial'}
layout = {'width': '600px'}

t_adc_slider = widgets.IntSlider(
    value=150, min=50, max=500, step=10, 
    description='ADC Settle Time (ns):', style=style, layout=layout)

t_opamp_slider = widgets.IntSlider(
    value=600, min=100, max=1500, step=50, 
    description='Op-Amp Settle Time (ns):', style=style, layout=layout)

t_margin_slider = widgets.IntSlider(
    value=300, min=0, max=1000, step=50, 
    description='Blanking + Margin Time (ns):', style=style, layout=layout)

f_sw_slider = widgets.FloatSlider(
    value=20, min=10, max=100, step=5, 
    description='PWM Frequency (kHz):', style=style, layout=layout)

# --- 3. Link Sliders to the Calculation Function ---
# Group controls and output for a clean layout
controls = widgets.VBox([t_adc_slider, t_opamp_slider, t_margin_slider, f_sw_slider])
out = widgets.interactive_output(calculate_limits, {
    't_adc_ns': t_adc_slider, 
    't_opamp_ns': t_opamp_slider,
    't_margin_ns': t_margin_slider,
    'f_sw_khz': f_sw_slider
})

# Display the final layout
display(HTML("<h2>Interactive Dual-Shunt Modulation Limit Calculator</h2>"))
display(controls, out)

Output()